# 🧪 Experiments
## Marketing Intelligence AI Platform

**Purpose:** Structured scratchpad for model experiments, hyperparameter sweeps, and ablation studies.
Track all results in the comparison table at the bottom before promoting to production.

**Sections:**
1. Environment Setup
2. Baseline Models
3. Hyperparameter Tuning — Optuna / GridSearchCV
4. Feature Ablation Study
5. Threshold Optimisation (Revenue Risk)
6. Anomaly Contamination Rate Sweep
7. K-Means Optimal-K Study
8. Cross-Validation Stability
9. Experiment Results Log

---
## 1. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import lightgbm as lgb
import joblib

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

# Experiment tracking log — appended throughout the notebook
EXPERIMENT_LOG = []

def log_experiment(name: str, params: dict, metrics: dict) -> None:
    """Append a result to the experiment log."""
    EXPERIMENT_LOG.append({
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'experiment': name,
        **params,
        **metrics,
    })
    print(f'[LOG] {name} → {metrics}')

print('✅ Environment ready. Experiment log initialised.')

---
## 2. Shared Synthetic Dataset

In [ ]:
# TODO: Replace with: df = DataLoader.load_feature_store()

np.random.seed(0)
N = 2000
N_FEATURES = 20

X = np.random.randn(N, N_FEATURES)
y = (np.random.rand(N) > 0.7).astype(int)   # binary: revenue drop flag

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Positive rate (train): {y_train.mean():.2%}')

---
## 3. Baseline Models

In [ ]:
# Dummy classifiers to establish baseline
for strategy in ['most_frequent', 'stratified', 'prior']:
    dummy = DummyClassifier(strategy=strategy)
    dummy.fit(X_train, y_train)
    proba = dummy.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    f1    = f1_score(y_test, dummy.predict(X_test))
    log_experiment('Baseline', {'strategy': strategy}, {'auc': round(auc,4), 'f1': round(f1,4)})

# Random Forest baseline
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
rf_f1  = f1_score(y_test, rf.predict(X_test))
log_experiment('RandomForest', {'n_estimators': 100}, {'auc': round(rf_auc,4), 'f1': round(rf_f1,4)})

---
## 4. Hyperparameter Tuning — XGBoost (GridSearchCV)

In [ ]:
# Small grid for demonstration — expand for production tuning
param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
}

xgb_base = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# TODO: Replace with Optuna for faster Bayesian tuning.
# grid_search = GridSearchCV(xgb_base, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
# grid_search.fit(X_train, y_train)
# best_params = grid_search.best_params_
# best_auc    = grid_search.best_score_

# Placeholder
best_params = {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
best_auc    = 0.0  # TODO: Update with real grid search result.

log_experiment('XGBoost-GridSearch', best_params, {'cv_auc': round(best_auc, 4)})
print('Best params:', best_params)

---
## 5. Optuna Hyperparameter Sweep (XGBoost)

**TODO:** Install `optuna` and uncomment the cell below.

In [ ]:
# TODO: Run Optuna sweep for XGBoost.
# pip install optuna
#
# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial):
#     params = {
#         'n_estimators':   trial.suggest_int('n_estimators', 50, 500),
#         'max_depth':      trial.suggest_int('max_depth', 3, 10),
#         'learning_rate':  trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'subsample':      trial.suggest_float('subsample', 0.5, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
#     }
#     model = xgb.XGBClassifier(**params, eval_metric='logloss', use_label_encoder=False)
#     scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
#     return scores.mean()
#
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=50, timeout=300)
# print('Best trial AUC:', study.best_value)
# print('Best params:', study.best_params)

print('TODO: Run Optuna sweep. Uncomment the cells above.')

---
## 6. Feature Ablation Study

In [ ]:
# Compare model performance with different feature subsets
FEATURE_GROUPS = {
    'all_features':       list(range(N_FEATURES)),
    'core_only':          list(range(5)),
    'core+rolling':       list(range(10)),
    'core+rolling+lags':  list(range(15)),
}

ablation_results = []
for group_name, cols in FEATURE_GROUPS.items():
    model = xgb.XGBClassifier(n_estimators=100, max_depth=4,
                               eval_metric='logloss', use_label_encoder=False)
    model.fit(X_train[:, cols], y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test[:, cols])[:,1])
    ablation_results.append({'feature_group': group_name, 'n_features': len(cols), 'auc': auc})
    log_experiment(f'Ablation-{group_name}', {'n_features': len(cols)}, {'auc': round(auc,4)})

ablation_df = pd.DataFrame(ablation_results)

fig = px.bar(ablation_df, x='feature_group', y='auc', text_auto='.3f',
             title='Feature Ablation — AUC-ROC by Feature Group',
             color='auc', color_continuous_scale='Blues')
fig.show()

---
## 7. Threshold Optimisation (Revenue Risk)

In [ ]:
from sklearn.metrics import precision_recall_curve

model = xgb.XGBClassifier(n_estimators=100, eval_metric='logloss', use_label_encoder=False)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)

best_idx       = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f'Optimal threshold : {best_threshold:.4f}')
print(f'F1 at threshold   : {best_f1:.4f}')

pr_df = pd.DataFrame({'threshold': thresholds, 'precision': precisions[:-1],
                       'recall': recalls[:-1], 'f1': f1_scores[:-1]})

fig = px.line(pr_df, x='threshold', y=['precision','recall','f1'],
              title='Precision, Recall & F1 vs Classification Threshold')
fig.add_vline(x=best_threshold, line_dash='dash', line_color='red',
              annotation_text=f'Best F1 threshold={best_threshold:.2f}')
fig.show()

log_experiment('Threshold-Opt', {}, {'best_threshold': round(best_threshold,4), 'best_f1': round(best_f1,4)})

---
## 8. Anomaly Contamination Rate Sweep

In [ ]:
contamination_rates = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
results = []

for rate in contamination_rates:
    iso = IsolationForest(contamination=rate, random_state=42)
    iso.fit(X_train)
    preds = iso.predict(X_test)
    n_anomalies = (preds == -1).sum()
    results.append({'contamination': rate, 'anomalies_detected': n_anomalies})
    log_experiment('IsoForest-ContaminationSweep', {'contamination': rate},
                   {'anomalies_detected': int(n_anomalies)})

sweep_df = pd.DataFrame(results)
fig = px.bar(sweep_df, x='contamination', y='anomalies_detected', text_auto=True,
             title='Anomalies Detected vs Contamination Rate',
             labels={'contamination':'Contamination Rate','anomalies_detected':'Anomalies in Test Set'})
fig.show()

---
## 9. K-Means Optimal-K Study (Elbow + Silhouette)

In [ ]:
k_range   = range(2, 12)
inertias  = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

k_df = pd.DataFrame({'k': list(k_range), 'inertia': inertias, 'silhouette': sil_scores})

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2, subplot_titles=['Elbow (Inertia)', 'Silhouette Score'])
fig.add_trace(go.Scatter(x=k_df.k, y=k_df.inertia, mode='lines+markers', name='Inertia'), row=1, col=1)
fig.add_trace(go.Scatter(x=k_df.k, y=k_df.silhouette, mode='lines+markers', name='Silhouette'), row=1, col=2)
fig.update_layout(title='K-Means — Elbow & Silhouette Analysis', showlegend=False)
fig.show()

best_k = k_df.loc[k_df.silhouette.idxmax(), 'k']
print(f'\n✅ Recommended K by Silhouette: {best_k}')

---
## 10. Cross-Validation Stability

In [ ]:
models_to_cv = {
    'XGBoost':   xgb.XGBClassifier(n_estimators=100, eval_metric='logloss', use_label_encoder=False),
    'LightGBM':  lgb.LGBMClassifier(n_estimators=100, verbose=-1),
    'RandomForest': RandomForestClassifier(n_estimators=100, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for model_name, model in models_to_cv.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'model': model_name,
        'mean_auc': scores.mean(),
        'std_auc':  scores.std(),
        'min_auc':  scores.min(),
        'max_auc':  scores.max(),
    })
    log_experiment(f'CV-{model_name}', {}, {'mean_auc': round(scores.mean(),4), 'std': round(scores.std(),4)})

cv_df = pd.DataFrame(cv_results)

fig = px.bar(cv_df, x='model', y='mean_auc', error_y='std_auc', text_auto='.3f',
             title='5-Fold CV AUC-ROC with Std Dev',
             color='mean_auc', color_continuous_scale='Blues')
fig.show()
cv_df

---
## 11. Experiment Results Log

In [ ]:
results_df = pd.DataFrame(EXPERIMENT_LOG)
print(f'Total experiments logged: {len(results_df)}')
results_df.style.background_gradient(subset=['auc'] if 'auc' in results_df.columns else [], cmap='Blues')

In [ ]:
# Save experiment log to CSV for reference
log_path = '../data/outputs/experiment_log.csv'
results_df.to_csv(log_path, index=False)
print(f'✅ Experiment log saved to {log_path}')

---
## Notes & Decisions

| Decision | Rationale | Status |
|---|---|---|
| XGBoost + LightGBM ensemble | Reduces variance vs single model | TODO: Validate on real data |
| Optimal threshold from PR curve | Better than default 0.5 for imbalanced classes | TODO |
| Contamination rate | Set from business domain knowledge | TODO |
| Optimal K for segmentation | Determined by Silhouette score | TODO |